# 07 - Classification And Decision Boundary

这本 `course` 不是压缩版提纲。课堂 notebook 会先把直觉、例子、代码和图跑通，再进入最后的 Predict / Modify 检查。

学习顺序：先读这一页的主线，遇到代码就运行；最后再做底部的小检查。真正写作业时进入同目录的 `homework.ipynb`。

In [ ]:
from pathlib import Path
import sys, math, random, inspect


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
PROJECT_ROOT = ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ModuleNotFoundError:
    plt = None
    MATPLOTLIB_AVAILABLE = False

try:
    import torch
except ModuleNotFoundError:
    torch = None


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def assert_close(actual, expected, name='value', eps=1e-9):
    assert abs(actual - expected) < eps, f'{name}: expected {expected}, got {actual}'


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

from course.checks import qa_check


def show_svg(path):
    path = Path(path)
    try:
        from IPython.display import SVG, display
        display(SVG(filename=str(path)))
    except Exception:
        print('图已生成，但当前环境不能内嵌显示。请打开:', path.resolve())

print('project root:', ROOT)


## 1. 分类不是直接输出类别，而是先输出 score

一个二分类模型可以先输出一个数字：

```text
score > 0 -> 预测 +1
score < 0 -> 预测 -1
```

这个 `score = 0` 的位置就是边界。二维平面里，它会变成一条线或一条弯曲边界。

In [ ]:
def label_from_score(score):
    return 1 if score > 0 else -1

for score in [-2.0, -0.1, 0.3, 4.0]:
    print(score, '->', label_from_score(score))

### 和以前学过的 `y = ax + b` 是同一件事

你可以把分类任务先理解成一个熟悉问题：

```text
以前：给你一些 (x, y) 点，找一条线 y = ax + b 去拟合它们。
现在：给你一些二维点 [x1, x2]，找一个 score 函数去分开红点和蓝点。
```

最简单的二维分类器可以写成：

```text
score = w1*x1 + w2*x2 + b
```

这和 `y = ax + b` 很像，只是输入从一个 `x` 变成了两个数 `[x1, x2]`：

```text
y = a*x + b              # 一维输入，输出一个 y
score = w1*x1 + w2*x2 + b # 二维输入，输出一个 score
```

分类时不直接拿 score 当答案，而是看它在 0 的哪一边：

```text
score > 0  预测蓝点，也就是 +1
score < 0  预测红点，也就是 -1
score = 0  就是分界线
```

所以本节最后那张图，本质上就是：**给模型一堆红蓝点，让它学出一条分界线**。

`MLP(2, [4, 1])` 只是把 `score = w1*x1 + w2*x2 + b` 变成更灵活的 score 函数，不再只能画一条很死的直线。



## 2. margin：不只要答对，还要答得有把握

如果真实标签 `y` 是 `1` 或 `-1`，我们看：

$$
margin = y \cdot score
$$

直觉：

| 情况 | margin | 意思 |
|---|---:|---|
| `y=1, score=2` | 2 | 答对，而且离边界远 |
| `y=1, score=0.2` | 0.2 | 答对，但很靠近边界 |
| `y=1, score=-1` | -1 | 答错 |

常用的 hinge loss 是：

$$
loss = max(0, 1 - margin)
$$

在 micrograd 里可以写成：

```python
(1 - y * score).relu()
```

In [ ]:
def hinge_loss(score, y):
    return (1 - y * score).relu()

examples = [
    (Value(2.0), 1),
    (Value(0.2), 1),
    (Value(-1.0), 1),
    (Value(-2.0), -1),
]

for score, y in examples:
    loss_value = hinge_loss(score, y)
    print(f'y={y:>2}, score={score.data:>4}, margin={y * score.data:>4}, loss={loss_value.data}')

## 3. 一个小分类数据集

这里每个点有两个输入 `x1, x2`，标签是 `1` 或 `-1`。

为了让 decision boundary 更直观，这里故意放多一点点：

```text
红点 y=-1：主要在左边
蓝点 y=+1：主要在右边
```

这样训练前先看散点图，就能先形成一个直觉：模型最后应该学出一条大概把左右两边分开的线。



In [ ]:
xs = [
    [-2.8, -1.8],
    [-2.6, -0.6],
    [-2.5,  1.0],
    [-2.2,  2.0],
    [-1.9, -2.2],
    [-1.6, -1.0],
    [-1.5,  0.8],
    [-1.2,  1.7],
    [-0.9, -1.6],
    [-0.7,  0.2],
    [-0.5,  1.1],
    [ 0.6, -1.2],
    [ 0.8,  0.6],
    [ 1.0,  1.8],
    [ 1.2, -2.0],
    [ 1.5, -0.4],
    [ 1.7,  1.2],
    [ 2.0, -1.5],
    [ 2.2,  0.4],
    [ 2.5,  1.6],
    [ 2.8, -0.7],
]
ys = [-1] * 11 + [1] * 10

for i, (x, y) in enumerate(zip(xs, ys)):
    print(f'{i:02d}: {x} -> {y:+d}')



### 先画原始数据：红点和蓝点到底是什么

这节课先把数据看成平面上的点：

```text
每个 x = [x1, x2] 是一个二维点
标签 y =  1 画成蓝点
标签 y = -1 画成红点
```

你先肉眼看：红点大多在左边，蓝点大多在右边。模型训练的目标，就是学出一条线，把红点和蓝点尽量分开。



In [ ]:
CLASS_COLORS = {1: '#2563eb', -1: '#dc2626'}
CLASS_NAMES = {1: 'blue: y=+1', -1: 'red: y=-1'}


def plot_points(ax, scores=None, preds=None, annotate=True):
    seen_labels = set()
    for i, (x, y) in enumerate(zip(xs, ys)):
        label = CLASS_NAMES[y] if y not in seen_labels else None
        seen_labels.add(y)
        ax.scatter(
            x[0], x[1],
            c=CLASS_COLORS[y],
            s=85,
            edgecolors='black',
            linewidths=1,
            label=label,
            zorder=3,
        )

        if annotate:
            note = str(i)
            if scores is not None and preds is not None:
                ok = 'OK' if preds[i] == y else 'MISS'
                note = f'{i} {ok}'
            ax.annotate(note, (x[0] + 0.06, x[1] + 0.06), fontsize=8)

    ax.axhline(0, color='#999999', linewidth=1, alpha=0.4)
    ax.axvline(0, color='#999999', linewidth=1, alpha=0.4)
    ax.set_xlim(-3.2, 3.2)
    ax.set_ylim(-2.8, 2.8)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.grid(True, alpha=0.2)
    ax.legend(loc='upper left')


if MATPLOTLIB_AVAILABLE:
    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    plot_points(ax)
    ax.set_title('Training data: red points vs blue points')
    plt.show()
else:
    print('matplotlib 不可用，跳过原始数据图。')



## 4. 训练一个 MLP 分类器

这里还是用 `MLP(2, [4, 1])`：

```text
输入 2 个数字 -> 隐藏层 4 个神经元 -> 输出 1 个 score
```

注意：输出层最后没有 ReLU，所以 score 可以是正数也可以是负数。

In [ ]:
random.seed(7)
model = MLP(2, [4, 1])
print(model)
print('parameters:', len(model.parameters()))


def predict_score(xrow):
    return model([Value(x) for x in xrow])


def classification_loss():
    scores = [predict_score(x) for x in xs]
    losses = [hinge_loss(score, y) for score, y in zip(scores, ys)]
    total = sum(losses) * (1.0 / len(losses))
    return total, scores

initial_loss, initial_scores = classification_loss()
print('initial loss:', initial_loss.data)
print('initial scores:', [round(s.data, 3) for s in initial_scores])

## 5. 跑训练循环

还是 06 那四步：

```text
forward -> zero_grad -> backward -> update
```

In [ ]:
learning_rate = 0.05
history = []

for step in range(120):
    total_loss, scores = classification_loss()
    history.append(total_loss.data)

    model.zero_grad()
    total_loss.backward()

    for p in model.parameters():
        p.data += -learning_rate * p.grad

    if step % 20 == 0 or step == 119:
        preds = [label_from_score(s.data) for s in scores]
        correct = sum(int(p == y) for p, y in zip(preds, ys))
        print(f'step {step:03d} loss={total_loss.data:.4f} acc={correct}/{len(ys)}')

final_loss, final_scores = classification_loss()
print('initial loss:', history[0])
print('final loss  :', final_loss.data)
assert final_loss.data < history[0]



## 6. 画出最终分类图：红点、蓝点、黑线

这张图是本节最重要的直觉图。

读图规则：

```text
蓝点：真实标签 y = +1
红点：真实标签 y = -1
蓝色背景：模型认为 score > 0，会预测 +1
红色背景：模型认为 score < 0，会预测 -1
黑线：score = 0，也就是 decision boundary
```

所以：

```text
蓝点落在蓝色区域：预测正确
红点落在红色区域：预测正确
点离黑线越远：模型越有把握，margin 越大
点靠近黑线：虽然可能答对，但把握不大
```



In [ ]:
if MATPLOTLIB_AVAILABLE:
    final_loss, final_scores = classification_loss()
    final_preds = [label_from_score(score.data) for score in final_scores]

    print('i | x | y_true | score | pred | result | margin')
    print('--|---|--------|-------|------|--------|-------')
    for i, (x, y, score, pred) in enumerate(zip(xs, ys, final_scores, final_preds)):
        result = 'OK' if pred == y else 'MISS'
        margin = y * score.data
        print(f'{i} | {x} | {y:+d} | {score.data:+.3f} | {pred:+d} | {result} | {margin:+.3f}')

    grid_x = [i / 10 for i in range(-35, 36)]
    grid_y = [i / 10 for i in range(-30, 31)]
    Z = []
    for yy in grid_y:
        row = []
        for xx in grid_x:
            row.append(predict_score([xx, yy]).data)
        Z.append(row)

    fig, ax = plt.subplots(figsize=(7.2, 5.2))

    # 背景颜色表示模型在这个区域会预测哪一类。
    ax.contourf(
        grid_x, grid_y, Z,
        levels=[-999, 0, 999],
        colors=['#fee2e2', '#dbeafe'],
        alpha=0.9,
    )

    # 黑线是 score=0；线的一边预测 -1，另一边预测 +1。
    boundary = ax.contour(grid_x, grid_y, Z, levels=[0], colors='black', linewidths=2)
    ax.clabel(boundary, inline=True, fontsize=9, fmt={0: 'score=0'})

    plot_points(ax, scores=final_scores, preds=final_preds)
    ax.set_title('Final classifier: data points and decision boundary')
    plt.show()
else:
    print('matplotlib 不可用，跳过 decision boundary 图。')



## What To Remember

这一节记住六句话：

```text
1. 分类模型先输出 score，不是直接输出文字类别。
2. score > 0 判为 +1，score < 0 判为 -1。
3. margin = y * score，margin 越大越有把握。
4. hinge loss = max(0, 1 - margin)。
5. decision boundary 就是 score = 0 的位置。
6. 图里蓝点/红点是真实标签，背景颜色和黑线是模型学出来的分类规则。
```



## 课堂检查：先预测，再改一行

这一段保留 `course` 的隐藏检查。你应该先自己填，再展开提示或答案。

## Predict - y*score

In [ ]:
# y=-1, score=-2
student_margin = None



qa_check('qa_check_class_07_predict', globals())

<details><summary>Show / Hide 本题提示</summary>

- 先算 `margin = y * score`：答对方向会变成正数；hinge loss 是 `relu(安全距离 - margin)`。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_margin` 填 `2`。

</details>

## Run - hinge loss 小表格

In [ ]:
from micrograd.engine import Value

def hinge(score, y):
    return (1 - y*score).relu()

for score, y in [(2,1), (0.2,1), (-1,1), (-2,-1)]:
    print('score=', score, 'y=', y, 'margin=', y*score, 'loss=', hinge(Value(score), y).data)

## Run - 画红点蓝点和边界线

In [ ]:
xs = [(-2,-1), (-1,-2), (-2,1), (-1,1), (1,2), (2,1), (1,-1), (2,-2)]
ys = [-1, -1, -1, -1, 1, 1, 1, 1]

def write_svg(path='class_07_boundary.svg'):
    width = height = 320
    def sx(x): return 40 + (x+3)/6*240
    def sy(y): return 280 - (y+3)/6*240
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">', '<rect width="100%" height="100%" fill="white"/>']
    parts.append(f'<line x1="{sx(-3)}" y1="{sy(-3)}" x2="{sx(3)}" y2="{sy(3)}" stroke="black"/>')
    for (x,y), label in zip(xs, ys):
        color = '#e74c3c' if label == 1 else '#3498db'
        parts.append(f'<circle cx="{sx(x):.1f}" cy="{sy(y):.1f}" r="6" fill="{color}"/>')
    parts.append('</svg>')
    Path(path).write_text('\n'.join(parts), encoding='utf-8')
    return path

svg_path = write_svg()
show_svg(svg_path)

## Modify - 改安全距离

In [ ]:
# safety margin 从 1 改到 2；y=1,score=1.5
student_loss_margin2 = None



qa_check('qa_check_class_07_modify', globals())

<details><summary>Show / Hide 本题提示</summary>

- 先算 `margin = y * score`：答对方向会变成正数；hinge loss 是 `relu(安全距离 - margin)`。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_loss_margin2` 约等于 `0.5`。

</details>